# 信号测试分析工作簿

本文档按照 `signal-test/draft.md` 的要求，构建从数据准备、技术指标与信号计算、三围栏标签评估到统计报告导出的完整流程。所有代码按步骤拆分，并在关键环节穿插说明。

## 流程概览
- 加载 2020-01-01 至 2025-08-31 期间的标的池行情，并预留 60 日缓冲
- 计算 KDJ、MACD、EMA、DMI、ATR 等技术指标，统一生成信号
- 定义 “t 日上穿且 t-2、t-3 日位于下方” 的上穿判定
- 在 horizon=5~15、ATR 倍数=1.0~3.0 的网格上执行三围栏标签
- 汇总每个信号的胜率、净胜次数，并绘制示例图与热力图
- 导出包含单信号章节与全局对比表格的 HTML 报告

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display, Markdown, HTML

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "axes.grid.which": "both",
    "grid.alpha": 0.25,
})

## 标的池与基础参数
使用 draft 中给出的行业划分构建标的池，并声明核心时间、网格与文件路径参数。

In [ ]:
def locate_data_dir() -> Path:
    search_roots = [Path.cwd(), *list(Path.cwd().parents[:3])]
    for root in search_roots:
        candidate = root / "data/qlib_us_selected/source"
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError("未找到 data/qlib_us_selected/source 目录，请先构建 Qlib 数据集。")


DATA_DIR = locate_data_dir()
PROJECT_ROOT = DATA_DIR.parents[2]
OUTPUT_DIR = (PROJECT_ROOT / "signal-test").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ASSET_DIR = OUTPUT_DIR / "report_assets"
ASSET_DIR.mkdir(parents=True, exist_ok=True)
ASSET_DIR_REL = Path("report_assets")
HTML_REPORT_PATH = OUTPUT_DIR / "signal_report.html"

ANALYSIS_START = pd.Timestamp("2020-01-01")
ANALYSIS_END = pd.Timestamp("2025-08-31")
BUFFER_DAYS = 60
PLOT_START = pd.Timestamp("2024-01-01")
PLOT_END = pd.Timestamp("2024-12-31")
HORIZONS = list(range(5, 16, 2))
ATR_MULTIPLIERS = [round(x, 1) for x in np.linspace(1.0, 3.0, num=11)]
REWARD_RATIO = 1.5
MAX_HORIZON = max(HORIZONS)

TICKER_GROUPS: Dict[str, Sequence[str]] = {
    "科技": [
        "NVDA", "TSM", "AAPL", "MSFT", "GOOGL", "AMZN", "SAP", "ANET", "AVGO", "IBM",
        "TSLA", "PLTR", "ADP", "ALAB", "ADI", "TXN", "MU", "QCOM", "ARM", "SNDK", "RELX", "IONQ",
    ],
    "金融": [
        "JPM", "MS", "KKR", "MAIN", "V", "AXP", "PGR", "ICE", "BN", "SPGI", "BX", "NDAQ", "ARES", "STT",
    ],
    "传统消费与工业": ["ABBV", "CAT", "RTX", "VST", "MNST", "MCD", "CVX", "HWM"],
    "ADR": ["DBSDY", "ABBNY", "TKOMY", "TOELY", "NTDOY", "SFTBY"],
}

available_files = {path.stem for path in DATA_DIR.glob("*.csv")}
active_symbols: List[str] = []
missing_symbols: List[str] = []
for group_symbols in TICKER_GROUPS.values():
    for ticker in group_symbols:
        if ticker in available_files:
            active_symbols.append(ticker)
        else:
            missing_symbols.append(ticker)
active_symbols = sorted(set(active_symbols))

print(f"数据目录: {DATA_DIR}")
print(f"可用标的: {len(active_symbols)} 个 / draft 总计 {sum(len(v) for v in TICKER_GROUPS.values())} 个")
if missing_symbols:
    print("未找到的标的:", ", ".join(sorted(set(missing_symbols))))
if not active_symbols:
    raise ValueError("标的池为空，无法继续。")

## 数据读取
从 CSV 中读取日频 OHLCV 数据，同时预留前后缓冲期以保证技术指标和标签的滚动窗口完整。

In [ ]:
def load_price_data(symbols: Iterable[str], start: pd.Timestamp, end: pd.Timestamp, buffer_days: int) -> Dict[str, pd.DataFrame]:
    frames: Dict[str, pd.DataFrame] = {}
    start_with_buffer = start - pd.Timedelta(days=buffer_days)
    end_with_buffer = end + pd.Timedelta(days=buffer_days)
    for symbol in symbols:
        path = DATA_DIR / f"{symbol}.csv"
        if not path.exists():
            continue
        df = pd.read_csv(path, parse_dates=["date"]).set_index("date").sort_index()
        df = df.loc[start_with_buffer:end_with_buffer, ["open", "high", "low", "close", "volume"]]
        frames[symbol] = df
    return frames

In [ ]:
raw_data = load_price_data(active_symbols, ANALYSIS_START, ANALYSIS_END, BUFFER_DAYS)
print(f"成功载入 {len(raw_data)} 只标的的数据。")
sample_symbol = next(iter(raw_data))
print(f"示例标的: {sample_symbol}")
display(raw_data[sample_symbol].head())

## 技术指标计算
复用常见公式生成 EMA、MACD、KDJ、DMI、ATR 等指标，为后续信号判定准备输入。

In [ ]:
@dataclass
class IndicatorParams:
    ema_periods: Tuple[int, ...] = (5, 10, 20, 60)
    macd_fast: int = 12
    macd_slow: int = 26
    macd_signal: int = 9
    kdj_period: int = 9
    kdj_smooth: int = 3
    dmi_period: int = 14
    atr_period: int = 14


def _ema(series: pd.Series, period: int) -> pd.Series:
    return series.ewm(span=period, adjust=False).mean()


def _macd(series: pd.Series, fast: int, slow: int, signal: int) -> pd.DataFrame:
    ema_fast = _ema(series, fast)
    ema_slow = _ema(series, slow)
    dif = ema_fast - ema_slow
    dea = dif.ewm(span=signal, adjust=False).mean()
    hist = (dif - dea) * 2
    return pd.DataFrame({"macd_dif": dif, "macd_dea": dea, "macd_hist": hist})


def _true_range(df: pd.DataFrame) -> pd.Series:
    prev_close = df["close"].shift(1)
    ranges = pd.concat(
        [
            df["high"] - df["low"],
            (df["high"] - prev_close).abs(),
            (df["low"] - prev_close).abs(),
        ],
        axis=1,
    )
    return ranges.max(axis=1)


def _atr(df: pd.DataFrame, period: int) -> pd.Series:
    return _true_range(df).rolling(window=period).mean()


def _dmi(df: pd.DataFrame, period: int) -> pd.DataFrame:
    tr = _true_range(df)
    tr_sum = tr.rolling(window=period).sum()
    up_move = df["high"].diff()
    down_move = -df["low"].diff()
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    plus_di = pd.Series(plus_dm, index=df.index).rolling(window=period).sum() * 100 / tr_sum
    minus_di = pd.Series(minus_dm, index=df.index).rolling(window=period).sum() * 100 / tr_sum
    dx = (plus_di - minus_di).abs() * 100 / (plus_di + minus_di)
    adx = dx.rolling(window=period).mean()
    adxr = (adx + adx.shift(period)) / 2
    return pd.DataFrame({"pdi": plus_di, "mdi": minus_di, "adx": adx, "adxr": adxr})


def _kdj(df: pd.DataFrame, period: int, smooth: int) -> pd.DataFrame:
    low_min = df["low"].rolling(window=period).min()
    high_max = df["high"].rolling(window=period).max()
    rsv = (df["close"] - low_min) / (high_max - low_min) * 100
    k = rsv.ewm(alpha=1 / smooth, adjust=False).mean()
    d = k.ewm(alpha=1 / smooth, adjust=False).mean()
    j = 3 * k - 2 * d
    return pd.DataFrame({"kdj_k": k, "kdj_d": d, "kdj_j": j})


def compute_indicator_panel(df: pd.DataFrame, params: IndicatorParams | None = None) -> pd.DataFrame:
    params = params or IndicatorParams()
    out = df.copy()
    for period in params.ema_periods:
        out[f"ema_{period}"] = _ema(out["close"], period)
    out = out.join(_macd(out["close"], params.macd_fast, params.macd_slow, params.macd_signal))
    out = out.join(_kdj(out, params.kdj_period, params.kdj_smooth))
    out = out.join(_dmi(out, params.dmi_period))
    out["atr"] = _atr(out, params.atr_period)
    return out

In [ ]:
indicator_frames = {symbol: compute_indicator_panel(df) for symbol, df in raw_data.items()}
print(f"完成指标计算: {len(indicator_frames)} 只标的")
display(indicator_frames[sample_symbol].head())

## 信号规则
按照 draft 中“t 日上穿 + t-2 / t-3 日仍位于下方”的定义生成 16 个信号，并实现趋势/阈值组合。

In [ ]:
SIGNAL_ORDER = [
    "KDJ",
    "KDJ-with-trend",
    "KDJ-with-limit",
    "KDJ-with-limit-trend",
    "MACD",
    "MACD-with-trend",
    "MACD-with-limit",
    "MACD-with-trend-limit",
    "EMA5-10",
    "EMA5-10-with-trend",
    "EMA5-20",
    "EMA5-20-with-trend",
    "ADX-ADXR",
    "ADX-ADXR-with-trend",
    "PDI-MDI",
    "PDI-MDI-with-trend",
]


def cross_over(series_a: pd.Series, series_b: pd.Series) -> pd.Series:
    cond = (
        (series_a > series_b)
        & (series_a.shift(2) < series_b.shift(2))
        & (series_a.shift(3) < series_b.shift(3))
    )
    return cond.fillna(False)


def build_signals(df: pd.DataFrame) -> pd.DataFrame:
    trend = df["ema_5"] > df["ema_60"]

    kdj_cross = cross_over(df["kdj_k"], df["kdj_d"])
    macd_cross = cross_over(df["macd_dif"], df["macd_dea"])
    ema5_10_cross = cross_over(df["ema_5"], df["ema_10"])
    ema5_20_cross = cross_over(df["ema_5"], df["ema_20"])
    adx_adxr_cross = cross_over(df["adx"], df["adxr"])
    pdi_mdi_cross = cross_over(df["pdi"], df["mdi"])

    signal_df = pd.DataFrame(index=df.index)
    signal_df["KDJ"] = kdj_cross
    signal_df["KDJ-with-trend"] = kdj_cross & trend
    signal_df["KDJ-with-limit"] = kdj_cross & (df["kdj_k"] < 50)
    signal_df["KDJ-with-limit-trend"] = signal_df["KDJ-with-limit"] & trend

    signal_df["MACD"] = macd_cross
    signal_df["MACD-with-trend"] = macd_cross & trend
    signal_df["MACD-with-limit"] = macd_cross & (df["macd_dif"] < 0)
    signal_df["MACD-with-trend-limit"] = signal_df["MACD-with-limit"] & trend

    signal_df["EMA5-10"] = ema5_10_cross
    signal_df["EMA5-10-with-trend"] = ema5_10_cross & trend
    signal_df["EMA5-20"] = ema5_20_cross
    signal_df["EMA5-20-with-trend"] = ema5_20_cross & trend

    signal_df["ADX-ADXR"] = adx_adxr_cross
    signal_df["ADX-ADXR-with-trend"] = adx_adxr_cross & trend
    signal_df["PDI-MDI"] = pdi_mdi_cross
    signal_df["PDI-MDI-with-trend"] = pdi_mdi_cross & trend

    return signal_df.astype(bool)

In [ ]:
signal_frames = {symbol: build_signals(df) for symbol, df in indicator_frames.items()}
full_frames = {
    symbol: indicator_frames[symbol].join(signal_frames[symbol], how="left")
    for symbol in indicator_frames
}
print(f"联合指标 + 信号数据准备完成，示例列: {list(full_frames[sample_symbol].columns)[:8]}")

## 三围栏标签计算
对每个信号触发事件，在 horizon ∈ [5,15]（步长 2）与 ATR 倍数 ∈ [1.0, 3.0]（步长 0.2）的网格上计算三围栏标签：
- 第一触发止盈（high ≥ 止盈价）记为 1
- 第一触发止损（low ≤ 止损价）记为 -1
- 在持有期内未触发则记为 0
若同日同时命中止盈/止损，优先视作止盈。

In [ ]:
def generate_events(
    frames: Dict[str, pd.DataFrame],
    signal_names: Iterable[str],
    horizons: List[int],
    atr_multipliers: List[float],
    reward_ratio: float,
    analysis_start: pd.Timestamp,
    analysis_end: pd.Timestamp,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    records: List[Tuple[int, str, str, pd.Timestamp, pd.Timestamp, int, float, int]] = []
    event_index: Dict[Tuple[str, str, pd.Timestamp], int] = {}
    next_event_id = 0
    max_horizon = max(horizons)

    for symbol, df in frames.items():
        close = df["close"].to_numpy()
        high = df["high"].to_numpy()
        low = df["low"].to_numpy()
        atr = df["atr"].to_numpy()
        index = df.index.to_list()

        for signal in signal_names:
            if signal not in df:
                continue
            signal_values = df[signal].fillna(False).to_numpy(dtype=bool)
            trigger_positions = np.where(signal_values)[0]
            if trigger_positions.size == 0:
                continue

            for pos in trigger_positions:
                entry_date = index[pos]
                if not (analysis_start <= entry_date <= analysis_end):
                    continue
                atr_value = atr[pos]
                if np.isnan(atr_value) or atr_value <= 0:
                    continue
                if pos + max_horizon >= len(df):
                    continue

                key = (signal, symbol, entry_date)
                if key not in event_index:
                    event_index[key] = next_event_id
                    next_event_id += 1
                event_id = event_index[key]

                entry_price = close[pos]
                future_highs = high[pos + 1 : pos + 1 + max_horizon]
                future_lows = low[pos + 1 : pos + 1 + max_horizon]
                future_dates = index[pos + 1 : pos + 1 + max_horizon]

                for horizon in horizons:
                    sub_highs = future_highs[:horizon]
                    sub_lows = future_lows[:horizon]
                    sub_dates = future_dates[:horizon]

                    for atr_mult in atr_multipliers:
                        stop_loss = entry_price - atr_value * atr_mult
                        take_profit = entry_price + atr_value * atr_mult * reward_ratio

                        hit_stop = np.where(sub_lows <= stop_loss)[0]
                        hit_take = np.where(sub_highs >= take_profit)[0]
                        first_stop = int(hit_stop[0]) if hit_stop.size else None
                        first_take = int(hit_take[0]) if hit_take.size else None

                        if first_stop is None and first_take is None:
                            label = 0
                            exit_date = sub_dates[-1]
                        elif first_take is None:
                            label = -1
                            exit_date = sub_dates[first_stop]
                        elif first_stop is None:
                            label = 1
                            exit_date = sub_dates[first_take]
                        else:
                            if first_take <= first_stop:
                                label = 1
                                exit_date = sub_dates[first_take]
                            else:
                                label = -1
                                exit_date = sub_dates[first_stop]

                        records.append(
                            (
                                event_id,
                                signal,
                                symbol,
                                entry_date,
                                exit_date,
                                horizon,
                                float(atr_mult),
                                int(label),
                            )
                        )

    events_df = pd.DataFrame(
        records,
        columns=[
            "event_id",
            "signal",
            "symbol",
            "entry_date",
            "exit_date",
            "horizon",
            "atr_mult",
            "label",
        ],
    )
    base_counts = pd.Series(event_index).reset_index(name="event_id")
    base_counts = base_counts.groupby("level_0").size().reset_index(name="unique_triggers")
    base_counts = base_counts.rename(columns={"level_0": "signal"})
    return events_df, base_counts

In [ ]:
%%time
events_df, base_trigger_counts = generate_events(
    frames=full_frames,
    signal_names=SIGNAL_ORDER,
    horizons=HORIZONS,
    atr_multipliers=ATR_MULTIPLIERS,
    reward_ratio=REWARD_RATIO,
    analysis_start=ANALYSIS_START,
    analysis_end=ANALYSIS_END,
)
print(events_df.shape)
display(events_df.head())

## 指标统计
基于事件数据统计胜率与净胜次数，分别寻找胜率 / 净胜次数最高的组合。

In [ ]:
def count_wins(series: pd.Series) -> int:
    return int((series == 1).sum())


def count_losses(series: pd.Series) -> int:
    return int((series == -1).sum())


metrics_df = (
    events_df.groupby(["signal", "horizon", "atr_mult"])
    .agg(
        trades=("label", "count"),
        wins=("label", count_wins),
        losses=("label", count_losses),
        net=("label", "sum"),
    )
    .reset_index()
)
metrics_df["win_rate"] = metrics_df["wins"] / metrics_df["trades"]

best_win_df = (
    metrics_df.sort_values(["signal", "win_rate", "net"], ascending=[True, False, False])
    .groupby("signal")
    .head(1)
    .reset_index(drop=True)
)

best_net_df = (
    metrics_df.sort_values(["signal", "net", "win_rate"], ascending=[True, False, False])
    .groupby("signal")
    .head(1)
    .reset_index(drop=True)
)

summary_win = best_win_df.merge(base_trigger_counts, on="signal", how="left")
summary_net = best_net_df.merge(base_trigger_counts, on="signal", how="left")

print("最佳胜率组合预览：")
display(summary_win.head())
print("最佳净胜组合预览：")
display(summary_net.head())

## 汇总表（Notebook 展示）

In [ ]:
display(summary_win.sort_values("win_rate", ascending=False).reset_index(drop=True))
display(summary_net.sort_values("net", ascending=False).reset_index(drop=True))

## 可视化与信号小结
- 固定案例：TSM，horizon=5，ATR×=2，展示 2024 年信号表现
- 热力图仅使用英文标题/坐标，并保存到 report_assets 目录

In [ ]:
DEMO_SYMBOL = "TSM"
DEMO_HORIZON = 5
DEMO_ATR_MULT = 2.0

best_win_lookup = summary_win.set_index("signal").to_dict("index")
best_net_lookup = summary_net.set_index("signal").to_dict("index")


def slugify(name: str) -> str:
    return name.lower().replace(" ", "-")


def plot_signal_demo(
    df: pd.DataFrame,
    events: pd.DataFrame,
    signal: str,
    symbol: str,
    horizon: int,
    atr_mult: float,
    save_path: Path | None = None,
) -> Path | None:
    plot_df = df.loc[PLOT_START:PLOT_END]
    if plot_df.empty:
        display(Markdown(f"> ⚠️ {symbol} has no data between {PLOT_START:%Y-%m-%d} and {PLOT_END:%Y-%m-%d}. Demo skipped."))
        return None

    mask = (
        (events["signal"] == signal)
        & (events["symbol"] == symbol)
        & (events["horizon"] == horizon)
        & (events["atr_mult"] == atr_mult)
        & (events["entry_date"] >= PLOT_START)
        & (events["entry_date"] <= PLOT_END)
    )
    event_slice = events.loc[mask].sort_values("entry_date")

    positions = np.arange(len(plot_df))
    fig, (ax_price, ax_macd, ax_kdj) = plt.subplots(
        3,
        1,
        figsize=(14, 9),
        sharex=True,
        gridspec_kw={"height_ratios": [3, 1.2, 1.2]},
    )
    candle_width = 0.6
    for idx, (_, row) in zip(positions, plot_df.iterrows()):
        color = "#ef4444" if row["close"] >= row["open"] else "#22c55e"
        ax_price.plot([idx, idx], [row["low"], row["high"]], color=color, linewidth=1)
        lower = min(row["open"], row["close"])
        height = max(row["open"], row["close"]) - lower or 1e-10
        ax_price.add_patch(Rectangle((idx - candle_width / 2, lower), candle_width, height, facecolor=color, edgecolor=color))

    ema_colors = {
        "ema_5": "#0ea5e9",
        "ema_10": "#f59e0b",
        "ema_20": "#10b981",
        "ema_60": "#6366f1",
    }
    for col, color in ema_colors.items():
        ax_price.plot(positions, plot_df[col].values, label=col.upper(), color=color, linewidth=1.4)

    color_map = {1: "#ef4444", 0: "#fbbf24", -1: "#22c55e"}
    if event_slice.empty:
        ax_price.text(0.02, 0.95, "No triggers during 2024", transform=ax_price.transAxes, fontsize=11, color="#64748b", va="top")
    else:
        for _, ev in event_slice.iterrows():
            if ev["entry_date"] not in plot_df.index:
                continue
            start_idx = plot_df.index.get_loc(ev["entry_date"])
            if ev["exit_date"] in plot_df.index:
                end_idx = plot_df.index.get_loc(ev["exit_date"])
            else:
                end_idx = len(plot_df) - 1
            for axis in (ax_price, ax_macd, ax_kdj):
                axis.axvspan(start_idx - 0.5, end_idx + 0.5, color=color_map[ev["label"]], alpha=0.18)

    ax_price.set_title(f"{symbol} — {signal} Demo (2024, horizon={horizon}, ATR={atr_mult:.1f})")
    ax_price.set_ylabel("Price")
    ax_price.legend(loc="upper left", ncol=2, fontsize=9)

    hist_colors = ["#ef4444" if val >= 0 else "#22c55e" for val in plot_df["macd_hist"]]
    ax_macd.bar(positions, plot_df["macd_hist"], color=hist_colors, alpha=0.6, width=0.8, label="MACD Hist")
    ax_macd.plot(positions, plot_df["macd_dif"], color="#0ea5e9", label="MACD DIF")
    ax_macd.plot(positions, plot_df["macd_dea"], color="#f97316", label="MACD DEA")
    ax_macd.set_ylabel("MACD")
    ax_macd.legend(loc="upper left", ncol=3, fontsize=9)

    ax_kdj.plot(positions, plot_df["kdj_k"], color="#0ea5e9", label="KDJ K")
    ax_kdj.plot(positions, plot_df["kdj_d"], color="#f97316", label="KDJ D")
    ax_kdj.plot(positions, plot_df["kdj_j"], color="#10b981", label="KDJ J")
    ax_kdj.set_ylabel("KDJ")
    ax_kdj.legend(loc="upper left", ncol=3, fontsize=9)

    tick_step = max(len(plot_df) // 10, 1)
    tick_positions = list(range(0, len(plot_df), tick_step))
    if tick_positions[-1] != len(plot_df) - 1:
        tick_positions.append(len(plot_df) - 1)
    ax_kdj.set_xticks(tick_positions)
    ax_kdj.set_xticklabels([plot_df.index[i].strftime("%Y-%m-%d") for i in tick_positions], rotation=45, ha="right")
    ax_kdj.set_xlabel("Date")

    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=140, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    return save_path


def plot_heatmap(
    matrix: pd.DataFrame,
    title: str,
    best_line: pd.Series,
    cmap: str,
    colorbar_label: str,
    save_path: Path | None = None,
) -> Path | None:
    if matrix.empty:
        display(Markdown("> ⚠️ No data available for heatmap."))
        return None

    matrix = matrix.sort_index().sort_index(axis=1)
    data = np.ma.masked_invalid(matrix.to_numpy(dtype=float))
    fig, ax = plt.subplots(figsize=(6.8, 4.8))
    im = ax.imshow(data, aspect="auto", origin="lower", cmap=cmap)
    ax.set_xticks(np.arange(len(matrix.columns)))
    ax.set_xticklabels(matrix.columns)
    ax.set_yticks(np.arange(len(matrix.index)))
    ax.set_yticklabels(matrix.index)
    ax.set_xlabel("Holding horizon (days)")
    ax.set_ylabel("ATR multiple")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label=colorbar_label)

    line_vals = best_line.reindex(matrix.columns)
    row_positions: List[float] = []
    atr_levels = list(matrix.index)
    for val in line_vals:
        if pd.isna(val):
            row_positions.append(np.nan)
        else:
            try:
                row_positions.append(float(atr_levels.index(val)))
            except ValueError:
                row_positions.append(np.nan)
    ax.plot(np.arange(len(matrix.columns)), row_positions, color="white", marker="o", linewidth=1.5)

    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=140, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    return save_path


assets: Dict[str, Dict[str, object]] = {}

for signal in SIGNAL_ORDER:
    if signal not in best_win_lookup:
        continue
    win_meta = best_win_lookup[signal]
    net_meta = best_net_lookup.get(signal, win_meta)
    trigger_count = int(win_meta.get("unique_triggers", 0))
    win_h = int(win_meta["horizon"])
    win_atr = float(win_meta["atr_mult"])
    win_rate = win_meta.get("win_rate", float("nan"))
    win_net = int(win_meta.get("net", 0))
    net_h = int(net_meta.get("horizon", win_h))
    net_atr = float(net_meta.get("atr_mult", win_atr))
    net_value = int(net_meta.get("net", win_net))

    display(Markdown(
        f"### {signal}
"
        f"- 触发次数：{trigger_count}
"
        f"- 胜率最佳：h={win_h}，ATR×={win_atr:.1f}，胜率 {win_rate:.2%}，净胜 {win_net}
"
        f"- 净胜最佳：h={net_h}，ATR×={net_atr:.1f}，净胜 {net_value}"
    ))

    frame = full_frames.get(DEMO_SYMBOL)
    slug = slugify(signal)
    demo_path = None
    if frame is None:
        display(Markdown(f"> ⚠️ Demo symbol {DEMO_SYMBOL} missing from dataset."))
    else:
        demo_path = plot_signal_demo(
            frame,
            events_df,
            signal,
            DEMO_SYMBOL,
            DEMO_HORIZON,
            DEMO_ATR_MULT,
            save_path=ASSET_DIR / f"{slug}_demo.png",
        )

    signal_metrics = metrics_df[metrics_df["signal"] == signal]
    if signal_metrics.empty:
        display(Markdown("> ⚠️ No statistics available; skip heatmaps."))
        continue

    win_matrix = signal_metrics.pivot(index="atr_mult", columns="horizon", values="win_rate")
    net_matrix = signal_metrics.pivot(index="atr_mult", columns="horizon", values="net")

    best_atr_by_h = win_matrix.idxmax()
    win_heatmap_path = plot_heatmap(
        win_matrix,
        f"{signal} Win Rate",
        best_atr_by_h,
        cmap="viridis",
        colorbar_label="Win rate",
        save_path=ASSET_DIR / f"{slug}_winrate.png",
    )

    best_net_by_h = net_matrix.idxmax()
    net_heatmap_path = plot_heatmap(
        net_matrix,
        f"{signal} Net Wins",
        best_net_by_h,
        cmap="RdYlGn",
        colorbar_label="Net wins",
        save_path=ASSET_DIR / f"{slug}_netwins.png",
    )

    assets[signal] = {
        "demo_path": demo_path,
        "win_heatmap": win_heatmap_path,
        "net_heatmap": net_heatmap_path,
        "win_rate": win_rate,
        "win_h": win_h,
        "win_atr": win_atr,
        "net_h": net_h,
        "net_atr": net_atr,
        "net_value": net_value,
        "unique_triggers": trigger_count,
    }


In [ ]:
best_win_lookup = summary_win.set_index("signal").to_dict("index")
best_net_lookup = summary_net.set_index("signal").to_dict("index")

summary_win_table = summary_win.sort_values("win_rate", ascending=False).to_html(index=False, float_format="{:.2f}".format)
summary_net_table = summary_net.sort_values("net", ascending=False).to_html(index=False, float_format="{:.2f}".format)

sections: List[str] = []
for signal in SIGNAL_ORDER:
    if signal not in assets:
        continue
    info = assets[signal]
    win_meta = best_win_lookup.get(signal, {})
    net_meta = best_net_lookup.get(signal, win_meta)

    def rel_href(path_obj):
        if isinstance(path_obj, Path):
            return (ASSET_DIR_REL / path_obj.name).as_posix()
        return None

    demo_href = rel_href(info.get("demo_path"))
    win_heatmap_href = rel_href(info.get("win_heatmap"))
    net_heatmap_href = rel_href(info.get("net_heatmap"))

    sections.append(
        f"""
        <section class="signal-block">
          <h2>{signal}</h2>
          <ul>
            <li>Triggers: {int(info.get('unique_triggers', 0))}</li>
            <li>Best win rate: horizon={int(info.get('win_h', 0))}, ATR×={float(info.get('win_atr', 0)):.1f}, win rate {info.get('win_rate', float('nan')):.2%}, net {int(win_meta.get('net', 0))}</li>
            <li>Best net wins: horizon={int(info.get('net_h', 0))}, ATR×={float(info.get('net_atr', 0)):.1f}, net {int(info.get('net_value', 0))}</li>
          </ul>
          {f'<figure><img src="{demo_href}" alt="{signal} demo"><figcaption>TSM demo (2024, horizon=5, ATR=2.0)</figcaption></figure>' if demo_href else ''}
          <div class="heatmaps">
            {f'<figure><img src="{win_heatmap_href}" alt="{signal} win rate"><figcaption>Win rate heatmap</figcaption></figure>' if win_heatmap_href else ''}
            {f'<figure><img src="{net_heatmap_href}" alt="{signal} net wins"><figcaption>Net wins heatmap</figcaption></figure>' if net_heatmap_href else ''}
          </div>
        </section>
        """
    )

html = f"""<!DOCTYPE html>
<html lang="zh">
<head>
  <meta charset="utf-8">
  <title>Signal Test Report</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 24px; color: #1f2937; }}
    h1 {{ margin-bottom: 10px; }}
    h2 {{ color: #2563eb; margin-top: 24px; }}
    table {{ border-collapse: collapse; margin: 16px 0; width: 100%; }}
    th, td {{ border: 1px solid #d1d5db; padding: 6px 10px; text-align: right; }}
    th {{ background-color: #f3f4f6; }}
    td:first-child, th:first-child {{ text-align: left; }}
    section.signal-block {{ border-top: 1px solid #e5e7eb; padding-top: 16px; margin-top: 24px; }}
    figure {{ margin: 12px 0; }}
    figure img {{ max-width: 100%; height: auto; border: 1px solid #e5e7eb; }}
    .heatmaps {{ display: flex; gap: 16px; flex-wrap: wrap; }}
    .heatmaps figure {{ flex: 1 1 260px; }}
  </style>
</head>
<body>
  <h1>Signal Test Report (2020-2025)</h1>
  <p>Universe size: {len(active_symbols)} symbols. Demo chart uses TSM with horizon=5 and ATR×=2.0.</p>
  <h2>Best Win Rate</h2>
  {summary_win_table}
  <h2>Best Net Wins</h2>
  {summary_net_table}
  {''.join(sections)}
</body>
</html>"""

HTML_REPORT_PATH.write_text(html, encoding="utf-8")
display(Markdown(f"报告已生成：[signal_report.html]({HTML_REPORT_PATH})"))


## 下一步
- 若需在 Notebook 中继续分析，可直接复用 `events_df` 与 `metrics_df`
- 可根据报告结果挑选感兴趣的组合进一步做回测或交易模拟